# LeetCode 410 — Split Array Largest Sum (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def splitArray(self, nums: List[int], k: int) -> int:

```

Hãy tưởng tượng bạn có một dãy thùng hàng xếp **theo đúng thứ tự** trên băng chuyền, cân nặng lần lượt là `7, 2, 5, 10, 8`. Bạn có `k = 2` chiếc xe tải. Mỗi xe phải chở một **đoạn liên tiếp** các thùng (không được đổi thứ tự, không được bỏ thùng nào). Xe nào chở nặng nhất thì sẽ tốn xăng nhất — bạn muốn chia sao cho **xe nặng nhất nhẹ nhất có thể**. Đó chính là bài 410.

Bài này là "bài kinh điển số một" để học kỹ thuật **binary search trên đáp án** (binary search on the answer) — một kỹ thuật cực kỳ hay gặp trong phỏng vấn với các đề dạng *"minimize the maximum"* (làm nhỏ nhất cái lớn nhất). Nhưng ta sẽ **không nhảy ngay** vào kỹ thuật đó: ta đi từ cách mà người mới tự nghĩ ra, thấy nó vướng ở đâu, rồi cải tiến từng bước.

---

## Sau notebook này, bạn sẽ nắm được

1. Hiểu đề bằng hình ảnh **"đặt vách ngăn"**: chia thành `k` đoạn = đặt `k - 1` vách vào `n - 1` khe.
2. Công cụ nền: `itertools.combinations`, **prefix sum** (tổng tiền tố) để tính tổng một đoạn trong `O(1)`.
3. **Cách 1 — Brute force**: thử mọi cách đặt vách, `O(C(n-1, k-1) · n)`. Dễ hiểu nhưng bùng nổ.
4. **Cách 2 — Đệ quy + memo** (`functools.lru_cache`): trạng thái `(i, m)` = "chia phần đuôi `nums[i:]` thành `m` đoạn", `O(k · n²)`.
5. **Cách 3 — DP bảng** (quy hoạch động 2 chiều, điền từ dưới lên), cũng `O(k · n²)`, cách tạo list 2 chiều đúng trong Python.
6. **Đổi câu hỏi**: thay vì *"đáp án nhỏ nhất là bao nhiêu?"* hãy hỏi *"đáp án có thể ≤ X không?"* — và kiểm tra câu đó bằng **tham lam** (greedy) trong `O(n)`.
7. **Tính đơn điệu** (monotonicity) và **binary search từ căn bản** → **Cách 4 (cách chính)**: `O(n · log(sum))`. Vì sao `lo = max(nums)` và `hi = sum(nums)`.
8. Trace từng bước, `class Solution`, test đối chiếu chéo cả 4 cách, vì sao đúng, lỗi hay gặp, mẹo nhận dạng, biến thể và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Given an integer array `nums` and an integer `k`, split `nums` into `k` **non-empty subarrays** such that the largest sum of any subarray is **minimized**.

Return *the minimized largest sum of the split*.

A **subarray** is a contiguous part of the array.

**Example 1**

```text
Input:  nums = [7,2,5,10,8], k = 2
Output: 18
Explanation: There are four ways to split nums into two subarrays.
The best way is to split it into [7,2,5] and [10,8], where the largest sum among the two subarrays is only 18.
```

**Example 2**

```text
Input:  nums = [1,2,3,4,5], k = 2
Output: 9
Explanation: There are four ways to split nums into two subarrays.
The best way is to split it into [1,2,3] and [4,5], where the largest sum among the two subarrays is only 9.
```

**Constraints**

- `1 <= nums.length <= 1000`
- `0 <= nums[i] <= 10^6`
- `1 <= k <= min(50, nums.length)`

---

**Dịch nhanh và những điều cần để ý:**

- **Subarray** (mảng con) = một đoạn **liên tiếp** của mảng. `[7,2,5]` là subarray của `[7,2,5,10,8]`; còn `[7,5]` thì **không** (bỏ mất số 2 ở giữa).
- Chia **đúng** `k` đoạn, mỗi đoạn **không rỗng** (non-empty), các đoạn ghép lại phải đúng bằng mảng ban đầu (không bỏ, không đổi chỗ phần tử nào).
- Với mỗi cách chia, ta nhìn **tổng của từng đoạn** và lấy cái **lớn nhất**. Trong tất cả các cách chia, ta muốn giá trị lớn nhất đó **nhỏ nhất**. Trả về con số đó (một số nguyên), **không** phải cách chia.
- `nums[i]` có thể bằng **0** nhưng **không âm** — chi tiết nhỏ này rất quan trọng cho cách 4 (sẽ nói ở mục 9).
- `k <= nums.length` nên luôn chia được (tệ nhất mỗi phần tử một đoạn). `k <= 50` — nhỏ; `n <= 1000`; tổng mảng tối đa `1000 · 10^6 = 10^9`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

examples = [
    ([7, 2, 5, 10, 8], 2, 18),
    ([1, 2, 3, 4, 5], 2, 9),
]
for nums, k, expected in examples:
    print(f"nums={nums}, k={k} -> mong đợi {expected}")

nums=[7, 2, 5, 10, 8], k=2 -> mong đợi 18
nums=[1, 2, 3, 4, 5], k=2 -> mong đợi 9


## 2. Hiểu đề bằng tay: liệt kê mọi cách chia Example 1

Với `nums = [7,2,5,10,8]` và `k = 2`, ta chỉ cần chọn **một chỗ cắt**. Có 5 phần tử nên có 4 "khe" giữa chúng → 4 cách chia (đúng như đề nói *"four ways"*):

| cắt sau phần tử | đoạn 1 | đoạn 2 | tổng 2 đoạn | lớn nhất |
|---|---|---|---|---:|
| `7` | `[7]` | `[2,5,10,8]` | 7, 25 | 25 |
| `2` | `[7,2]` | `[5,10,8]` | 9, 23 | 23 |
| `5` | `[7,2,5]` | `[10,8]` | 14, 18 | **18** ← nhỏ nhất |
| `10` | `[7,2,5,10]` | `[8]` | 24, 8 | 24 |

Nhận xét trực giác: cắt quá sớm thì đoạn sau quá nặng; cắt quá muộn thì đoạn đầu quá nặng. Ta muốn **"cân bằng"** các đoạn. Tổng cả mảng là `32`, nếu chia đều tuyệt đối thì mỗi đoạn `16` — nhưng các phần tử không chẻ nhỏ được, nên tốt nhất chỉ đạt `18`.

Cell dưới làm lại bảng trên bằng code, dùng **slicing**: `nums[:c]` là các phần tử từ đầu tới **trước** chỉ số `c`; `nums[c:]` là từ chỉ số `c` tới hết.

In [2]:
nums = [7, 2, 5, 10, 8]
for c in range(1, len(nums)):          # c = vị trí cắt: đoạn 1 = nums[:c], đoạn 2 = nums[c:]
    left, right = nums[:c], nums[c:]
    print(f"c={c}: {str(left):<14} {str(right):<14} tổng={sum(left):>2},{sum(right):>2}  lớn nhất={max(sum(left), sum(right))}")

c=1: [7]            [2, 5, 10, 8]  tổng= 7,25  lớn nhất=25
c=2: [7, 2]         [5, 10, 8]     tổng= 9,23  lớn nhất=23
c=3: [7, 2, 5]      [10, 8]        tổng=14,18  lớn nhất=18
c=4: [7, 2, 5, 10]  [8]            tổng=24, 8  lớn nhất=24


## 3. Nền tảng 1: "vách ngăn" và `itertools.combinations`

Khi `k = 2` ta chỉ chọn 1 chỗ cắt. Tổng quát: chia mảng `n` phần tử thành `k` đoạn **tương đương** với việc đặt `k - 1` **vách ngăn** vào `n - 1` khe giữa các phần tử, mỗi khe tối đa một vách:

```text
nums :   7 | 2   5 | 10   8        (k = 3, vách ở khe 1 và khe 3)
khe  :     1   2   3    4
đoạn : [7] [2,5] [10,8]
```

- Vách ở khe `c` nghĩa là "cắt trước chỉ số `c`".
- Mỗi tập vách khác nhau cho một cách chia khác nhau, và ngược lại. Vì mỗi khe chỉ đặt tối đa 1 vách nên mọi đoạn đều **không rỗng**.
- Số cách chia = số cách chọn `k - 1` khe trong `n - 1` khe = **tổ hợp** `C(n-1, k-1)`.

### Kiến thức Python: `itertools.combinations(iterable, r)`

Hàm này sinh ra **mọi bộ `r` phần tử** lấy từ `iterable`, theo thứ tự tăng dần của vị trí, không lặp lại. Ví dụ `combinations([1,2,3,4], 2)` cho `(1,2), (1,3), (1,4), (2,3), (2,4), (3,4)` — đúng `C(4,2) = 6` bộ. `math.comb(n, r)` tính nhanh con số `C(n, r)`.

In [3]:
from itertools import combinations
import math

print(list(combinations([1, 2, 3, 4], 2)))
print("math.comb(4, 2) =", math.comb(4, 2))

# Áp vào bài: nums có 5 phần tử -> khe 1..4; k = 3 -> chọn 2 khe
nums = [7, 2, 5, 10, 8]
for cuts in combinations(range(1, len(nums)), 2):
    bounds = (0,) + cuts + (len(nums),)                      # thêm "vách" ở hai đầu
    parts = [nums[bounds[t]:bounds[t + 1]] for t in range(len(bounds) - 1)]
    print(cuts, "->", parts)

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
math.comb(4, 2) = 6
(1, 2) -> [[7], [2], [5, 10, 8]]
(1, 3) -> [[7], [2, 5], [10, 8]]
(1, 4) -> [[7], [2, 5, 10], [8]]
(2, 3) -> [[7, 2], [5], [10, 8]]
(2, 4) -> [[7, 2], [5, 10], [8]]
(3, 4) -> [[7, 2, 5], [10], [8]]


Đọc kỹ dòng `bounds = (0,) + cuts + (len(nums),)`: ta thêm vách "ảo" ở vị trí `0` (đầu mảng) và `n` (cuối mảng). Khi đó đoạn thứ `t` chính là `nums[bounds[t] : bounds[t+1]]`. Ví dụ `cuts = (1, 3)` → `bounds = (0, 1, 3, 5)` → đoạn `nums[0:1] = [7]`, `nums[1:3] = [2,5]`, `nums[3:5] = [10,8]`.

## 4. Nền tảng 2: prefix sum (tổng tiền tố)

Cả 4 cách giải đều phải tính **tổng một đoạn** rất nhiều lần. Gọi `sum(nums[a:b])` mỗi lần tốn `O(b - a)`. Mẹo kinh điển: tính trước mảng `P` với

- `P[0] = 0`
- `P[j] = nums[0] + nums[1] + ... + nums[j-1]` (tổng của `j` phần tử **đầu tiên**)

Khi đó **tổng đoạn `nums[a:b]` = `P[b] - P[a]`**, chỉ một phép trừ, `O(1)`.

Ví dụ `nums = [7,2,5,10,8]` → `P = [0, 7, 9, 14, 24, 32]`. Tổng `nums[1:4] = [2,5,10]` = `P[4] - P[1] = 24 - 7 = 17`. Đúng: `2 + 5 + 10 = 17`.

Vì sao `P` có `n + 1` phần tử và bắt đầu bằng `0`? Để công thức `P[b] - P[a]` đúng cả khi `a = 0` (đoạn bắt đầu từ đầu mảng): `P[b] - P[0] = P[b]`.

`itertools.accumulate(nums, initial=0)` tạo đúng mảng `P` này (tham số `initial=0` có từ Python 3.8).

In [4]:
from itertools import accumulate

nums = [7, 2, 5, 10, 8]
P = [0]
for x in nums:                 # cách tự viết
    P.append(P[-1] + x)
print("P tự viết   :", P)
print("P accumulate:", list(accumulate(nums, initial=0)))

a, b = 1, 4
print(f"tổng nums[{a}:{b}] = {nums[a:b]} =", sum(nums[a:b]), "| P[b]-P[a] =", P[b] - P[a])

P tự viết   : [0, 7, 9, 14, 24, 32]
P accumulate: [0, 7, 9, 14, 24, 32]
tổng nums[1:4] = [2, 5, 10] = 17 | P[b]-P[a] = 17


## 5. Cách 1 — Brute force: thử mọi cách đặt vách

Đây là cách người mới tự nghĩ ra ngay sau khi hiểu hình ảnh vách ngăn:

1. Duyệt mọi tập `k - 1` khe bằng `combinations(range(1, n), k - 1)`.
2. Với mỗi tập, tính tổng từng đoạn (dùng prefix sum) và lấy **max**.
3. Giữ lại **min** của các max đó.

Trường hợp `k = 1`: `combinations(..., 0)` sinh đúng **một** bộ rỗng `()` → chỉ có một đoạn là cả mảng. Code tự xử lý được, không cần `if` riêng.

Ngoài hàm trả về đáp án, ta viết thêm hàm trả về **cách chia tốt nhất** để nhìn tận mắt.

In [5]:
def split_array_bruteforce(nums, k):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = float("inf")
    for cuts in combinations(range(1, n), k - 1):
        bounds = (0,) + cuts + (n,)
        largest = max(P[bounds[t + 1]] - P[bounds[t]] for t in range(k))
        best = min(best, largest)
    return best


def best_split_bruteforce(nums, k):
    """Trả về (đáp án, một cách chia đạt đáp án) — chỉ để quan sát."""
    n = len(nums)
    best, best_parts = float("inf"), None
    for cuts in combinations(range(1, n), k - 1):
        bounds = (0,) + cuts + (n,)
        parts = [nums[bounds[t]:bounds[t + 1]] for t in range(k)]
        largest = max(sum(p) for p in parts)
        if largest < best:
            best, best_parts = largest, parts
    return best, best_parts


for nums, k, expected in examples:
    print(nums, k, "->", split_array_bruteforce(nums, k), "| mong đợi", expected, "|", best_split_bruteforce(nums, k))
print([7, 2, 5, 10, 8], 3, "->", best_split_bruteforce([7, 2, 5, 10, 8], 3))

[7, 2, 5, 10, 8] 2 -> 18 | mong đợi 18 | (18, [[7, 2, 5], [10, 8]])
[1, 2, 3, 4, 5] 2 -> 9 | mong đợi 9 | (9, [[1, 2, 3], [4, 5]])
[7, 2, 5, 10, 8] 3 -> (14, [[7, 2, 5], [10], [8]])


### Cách 1 vướng ở đâu?

**Độ phức tạp:** có `C(n-1, k-1)` cách đặt vách, mỗi cách tính `k` tổng đoạn (và tạo tuple `bounds` dài `k+1`) → Time `O(C(n-1, k-1) · k)`, nếu tính tổng bằng `sum` thì `O(C(n-1, k-1) · n)`. Space `O(n)` cho prefix sum.

Nghe thì vô hại, nhưng `C(n-1, k-1)` tăng **khủng khiếp**. Cell dưới in thử vài con số, và con số ở giới hạn đề (`n = 1000`, `k = 50`). Chỉ cần `C` vượt khoảng `10^7` là Python đã chạy mất vài giây.

In [6]:
for n, k in [(5, 2), (10, 3), (20, 5), (30, 10), (100, 10), (1000, 50)]:
    c = math.comb(n - 1, k - 1)
    print(f"n={n:>4}, k={k:>2}: C(n-1,k-1) = {c:.3e}")

n=   5, k= 2: C(n-1,k-1) = 4.000e+00
n=  10, k= 3: C(n-1,k-1) = 3.600e+01
n=  20, k= 5: C(n-1,k-1) = 3.876e+03
n=  30, k=10: C(n-1,k-1) = 1.002e+07
n= 100, k=10: C(n-1,k-1) = 1.731e+12
n=1000, k=50: C(n-1,k-1) = 4.730e+83


`C(999, 49)` khoảng `4.7 · 10^83` — nhiều hơn cả số nguyên tử trong vũ trụ quan sát được (khoảng `10^80`). Brute force **chỉ dùng được làm "trọng tài"** để kiểm tra các cách khác trên input nhỏ.

**Nút thắt nằm ở đâu?** Nhìn lại: cách chia `[7] [2,5] [10,8]` và `[7,2] [5] [10,8]` có chung đoạn cuối `[10,8]`. Brute force tính đi tính lại những "phần đuôi" giống nhau rất nhiều lần. Đây là dấu hiệu kinh điển của **bài toán con chồng lấp** (overlapping subproblems) → nghĩ tới **đệ quy + memo**.

## 6. Nền tảng 3: đệ quy (recursion) qua một bài nhỏ

**Đệ quy** = một hàm tự gọi lại chính nó trên một bài **nhỏ hơn**, cho tới khi gặp **trường hợp cơ sở** (base case) trả lời được ngay.

Bài khởi động: *"Có bao nhiêu cách chia phần đuôi `nums[i:]` thành `m` đoạn không rỗng?"* Gọi số đó là `ways(i, m)`.

- **Base case:** `m == 1` → chỉ có 1 cách (cả phần đuôi là một đoạn).
- **Bước đệ quy:** chọn **đoạn đầu tiên** là `nums[i:j]` (với `j` từ `i+1`), phần còn lại `nums[j:]` phải chia thành `m - 1` đoạn → `ways(j, m - 1)` cách. Cộng tất cả lại.
- Giới hạn `j`: phần còn lại cần ít nhất `m - 1` phần tử, nên `j <= n - (m - 1)`.

Đáp án phải bằng `C(n-i-1, m-1)` — ta dùng nó để kiểm tra. Đồng thời ta **đếm số lần hàm được gọi** để thấy vấn đề.

In [7]:
calls = 0

def ways(i, m, n):
    global calls
    calls += 1
    if m == 1:
        return 1
    total = 0
    for j in range(i + 1, n - (m - 1) + 1):   # đoạn đầu = nums[i:j]
        total += ways(j, m - 1, n)
    return total


for n, m in [(5, 2), (12, 4), (20, 6)]:
    calls = 0
    w = ways(0, m, n)
    print(f"n={n}, m={m}: ways={w}, C(n-1,m-1)={math.comb(n - 1, m - 1)}, số lần gọi hàm={calls}")

n=5, m=2: ways=4, C(n-1,m-1)=4, số lần gọi hàm=5
n=12, m=4: ways=165, C(n-1,m-1)=165, số lần gọi hàm=220
n=20, m=6: ways=11628, C(n-1,m-1)=11628, số lần gọi hàm=15504


Số lần gọi hàm tăng gần bằng chính số cách chia — vì cùng một câu hỏi `ways(j, m-1)` bị hỏi lại nhiều lần. Nhưng **số câu hỏi khác nhau** chỉ có tối đa `n · m` (mỗi cặp `(i, m)` một câu). Nếu ta **ghi nhớ** (memoize) câu trả lời của mỗi cặp, lần hỏi sau chỉ việc tra lại.

### Kiến thức Python: `functools.lru_cache`

Đặt `@lru_cache(maxsize=None)` ngay trên `def` ("decorator" — lớp bọc ngoài hàm): Python tự tạo một dict ẩn `(tham số) -> kết quả`. Lần đầu gọi với bộ tham số nào thì chạy thật và lưu lại; các lần sau trả kết quả đã lưu ngay lập tức. Điều kiện: tham số phải **hashable** (số, chuỗi, tuple — không phải list).

`maxsize=None` = không giới hạn số kết quả lưu. `ham.cache_info()` cho biết bao nhiêu lần trúng cache (`hits`) và bao nhiêu lần phải tính (`misses`).

In [8]:
from functools import lru_cache

@lru_cache(maxsize=None)
def ways_memo(i, m, n):
    if m == 1:
        return 1
    return sum(ways_memo(j, m - 1, n) for j in range(i + 1, n - (m - 1) + 1))


print("ways_memo(0, 6, 20) =", ways_memo(0, 6, 20))
print(ways_memo.cache_info())      # misses = số trạng thái thật sự phải tính

ways_memo(0, 6, 20) = 11628
CacheInfo(hits=420, misses=76, maxsize=None, currsize=76)


## 7. Cách 2 — Đệ quy + memo trên trạng thái `(i, m)`

Giờ thay "đếm số cách" bằng câu hỏi thật của bài:

> `f(i, m)` = **giá trị "đoạn lớn nhất" nhỏ nhất** khi chia phần đuôi `nums[i:]` thành `m` đoạn không rỗng.

Đáp án bài = `f(0, k)`.

- **Base case:** `m == 1` → chỉ một đoạn là cả phần đuôi: `f(i, 1) = P[n] - P[i]`.
- **Bước đệ quy:** thử mọi đoạn đầu `nums[i:j]` (tổng `P[j] - P[i]`). Với lựa chọn đó, đoạn lớn nhất của cả cách chia là `max(P[j] - P[i], f(j, m - 1))` — hoặc đoạn đầu là lớn nhất, hoặc đoạn lớn nhất nằm trong phần còn lại. Ta chọn `j` cho giá trị này **nhỏ nhất**:

```text
f(i, m) = min over j in [i+1, n-m+1] of  max( P[j] - P[i],  f(j, m-1) )
```

(`j <= n - m + 1` để phần còn lại có ít nhất `m - 1` phần tử.)

**Ví dụ số** với Example 1 (`P = [0,7,9,14,24,32]`, `n = 5`), tính `f(0, 2)`:

| đoạn đầu `nums[0:j]` | tổng đoạn đầu | `f(j, 1)` = tổng phần còn lại | max |
|---|---:|---:|---:|
| `j=1`: `[7]` | 7 | 25 | 25 |
| `j=2`: `[7,2]` | 9 | 23 | 23 |
| `j=3`: `[7,2,5]` | 14 | 18 | **18** |
| `j=4`: `[7,2,5,10]` | 24 | 8 | 24 |

→ `f(0, 2) = 18`. Khớp đề. Chú ý: đây chính là bảng ở mục 2, nhưng giờ được phát biểu thành công thức tổng quát.

In [9]:
def split_array_memo(nums, k):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)

    @lru_cache(maxsize=None)
    def f(i, m):
        if m == 1:
            return P[n] - P[i]
        best = float("inf")
        for j in range(i + 1, n - m + 2):          # j chạy tới n-m+1 (range loại trừ đầu mút phải)
            best = min(best, max(P[j] - P[i], f(j, m - 1)))
        return best

    return f(0, k)


for nums, k, expected in examples:
    print(nums, k, "->", split_array_memo(nums, k), "| mong đợi", expected)
print([7, 2, 5, 10, 8], 3, "->", split_array_memo([7, 2, 5, 10, 8], 3))

[7, 2, 5, 10, 8] 2 -> 18 | mong đợi 18
[1, 2, 3, 4, 5] 2 -> 9 | mong đợi 9
[7, 2, 5, 10, 8] 3 -> 14


### Trace cách 2: thấy memo hoạt động

Hàm `trace_memo` dưới đây làm y hệt `split_array_memo`, nhưng tự quản lý dict `memo` để in ra mỗi khi **tính mới** một trạng thái, hoặc **dùng lại** kết quả đã lưu. Thụt lề theo độ sâu đệ quy. Input: `[7,2,5,10,8]`, `k = 3` (đáp án `14`, chia `[7,2,5] [10] [8]`).

Để ý các dòng "dùng lại": `f(3,1)` và `f(4,1)` được hỏi nhiều lần nhưng chỉ tính một lần.

In [10]:
def trace_memo(nums, k):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    memo = {}

    def f(i, m, depth):
        pad = "    " * depth
        if (i, m) in memo:
            print(f"{pad}f({i},{m}) dùng lại memo = {memo[(i, m)]}")
            return memo[(i, m)]
        if m == 1:
            res = P[n] - P[i]
            print(f"{pad}f({i},{m}) base: tổng {nums[i:]} = {res}")
        else:
            res = float("inf")
            for j in range(i + 1, n - m + 2):
                res = min(res, max(P[j] - P[i], f(j, m - 1, depth + 1)))
            print(f"{pad}f({i},{m}) = {res}")
        memo[(i, m)] = res
        return res

    ans = f(0, k, 0)
    print("Đáp án:", ans, "| số trạng thái đã lưu:", len(memo))
    return ans


trace_memo([7, 2, 5, 10, 8], 3)

        f(2,1) base: tổng [5, 10, 8] = 23
        f(3,1) base: tổng [10, 8] = 18
        f(4,1) base: tổng [8] = 8
    f(1,2) = 17
        f(3,1) dùng lại memo = 18
        f(4,1) dùng lại memo = 8
    f(2,2) = 15
        f(4,1) dùng lại memo = 8
    f(3,2) = 10
f(0,3) = 14
Đáp án: 14 | số trạng thái đã lưu: 7


14

### Độ phức tạp cách 2

- **Số trạng thái:** `i` từ `0..n-1`, `m` từ `1..k` → tối đa `n · k` trạng thái.
- **Công việc mỗi trạng thái:** vòng `for j` tối đa `n` lần, mỗi lần `O(1)` nhờ prefix sum (và tra memo).
- **Time `O(k · n²)`**, **Space `O(k · n)`** cho memo (+ độ sâu đệ quy tối đa `k <= 50`, an toàn với giới hạn đệ quy mặc định 1000 của Python).

So với brute force là một bước nhảy vọt: từ `10^83` xuống `50 · 1000² = 5 · 10^7`. Nhưng `5 · 10^7` phép tính **bằng Python** vẫn là rất chậm (cỡ chục giây trở lên) — trên LeetCode dễ bị TLE (Time Limit Exceeded). Ta sẽ cần một ý tưởng khác hẳn. Trước đó, hãy viết lại cách 2 dưới dạng **bảng** — đây là kỹ năng DP cơ bản cần có.

## 8. Cách 3 — DP bảng (bottom-up)

Thay vì đệ quy "từ trên xuống" rồi nhớ, ta **điền bảng từ dưới lên** (bottom-up). Định nghĩa theo **tiền tố** (cho quen với cách viết DP phổ biến):

> `dp[m][j]` = đoạn lớn nhất nhỏ nhất khi chia **`j` phần tử đầu tiên** `nums[:j]` thành `m` đoạn.

- `dp[1][j] = P[j]` (một đoạn = cả tiền tố).
- `dp[m][j] = min over t in [m-1, j-1] of max( dp[m-1][t], P[j] - P[t] )` — **đoạn cuối** là `nums[t:j]`, phần trước `nums[:t]` chia thành `m-1` đoạn (cần `t >= m-1` phần tử).
- Ô không hợp lệ (`j < m`, không đủ phần tử) để `∞`. Đáp án: `dp[k][n]`.

Thứ tự điền: hàng `m` chỉ cần hàng `m-1` → điền lần lượt `m = 1, 2, ..., k`.

### Kiến thức Python: tạo list 2 chiều cho đúng

Cạm bẫy nổi tiếng: `[[0] * 3] * 2` **không** tạo 2 hàng độc lập mà tạo 2 **tham chiếu tới cùng một hàng**. Sửa một ô thì "cả cột" đổi theo. Cách đúng là dùng list comprehension: `[[0] * 3 for _ in range(2)]` — mỗi vòng lặp tạo một hàng **mới**.

In [11]:
bad = [[0] * 3] * 2
bad[0][0] = 99
print("Sai :", bad, "| hai hàng là cùng một object?", bad[0] is bad[1])

good = [[0] * 3 for _ in range(2)]
good[0][0] = 99
print("Đúng:", good, "| hai hàng là cùng một object?", good[0] is good[1])

Sai : [[99, 0, 0], [99, 0, 0]] | hai hàng là cùng một object? True
Đúng: [[99, 0, 0], [0, 0, 0]] | hai hàng là cùng một object? False


In [12]:
INF = float("inf")

def split_array_dp(nums, k, return_table=False):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    dp = [[INF] * (n + 1) for _ in range(k + 1)]    # dp[m][j], m = 0..k, j = 0..n
    for j in range(1, n + 1):
        dp[1][j] = P[j]
    for m in range(2, k + 1):
        for j in range(m, n + 1):                   # cần j >= m phần tử
            best = INF
            for t in range(m - 1, j):               # đoạn cuối = nums[t:j]
                best = min(best, max(dp[m - 1][t], P[j] - P[t]))
            dp[m][j] = best
    return (dp[k][n], dp) if return_table else dp[k][n]


for nums, k, expected in examples:
    print(nums, k, "->", split_array_dp(nums, k), "| mong đợi", expected)

ans, table = split_array_dp([7, 2, 5, 10, 8], 2, return_table=True)
print("\nBảng dp cho [7,2,5,10,8], k=2   (cột j = số phần tử đầu tiên)")
print("      " + "".join(f"j={j:<5}" for j in range(6)))
for m in range(1, 3):
    print(f"m={m}:  " + "".join(f"{('∞' if v == INF else v)!s:<7}" for v in table[m]))

[7, 2, 5, 10, 8] 2 -> 18 | mong đợi 18
[1, 2, 3, 4, 5] 2 -> 9 | mong đợi 9

Bảng dp cho [7,2,5,10,8], k=2   (cột j = số phần tử đầu tiên)
      j=0    j=1    j=2    j=3    j=4    j=5    
m=1:  ∞      7      9      14     24     32     
m=2:  ∞      ∞      7      7      14     18     


Đọc bảng (khớp với output ở trên):

| | j=0 | j=1 | j=2 | j=3 | j=4 | j=5 |
|---|---|---|---|---|---|---|
| m=1 | ∞ | 7 | 9 | 14 | 24 | 32 |
| m=2 | ∞ | ∞ | 7 | 7 | 14 | **18** |

Ví dụ ô `dp[2][4]` (chia `[7,2,5,10]` thành 2 đoạn): thử đoạn cuối `nums[1:4]=[2,5,10]` → `max(7, 17) = 17`; `nums[2:4]=[5,10]` → `max(9, 15) = 15`; `nums[3:4]=[10]` → `max(14, 10) = 14`. Nhỏ nhất `14`.

(`dp[m][0]` để `∞` vì không thể chia 0 phần tử thành đoạn không rỗng; hàng `m = 0` không dùng tới.)

**Độ phức tạp cách 3:** ba vòng lặp lồng `m`, `j`, `t` → **Time `O(k · n²)`**, **Space `O(k · n)`** (có thể giảm còn `O(n)` vì hàng `m` chỉ cần hàng `m-1`). Không còn đệ quy, nhưng vẫn `5 · 10^7` phép tính ở giới hạn đề.

> **Cách 2 và 3 vướng ở đâu?** Chúng trả lời một câu hỏi rất "giàu thông tin": *đáp án tối ưu cho MỌI tiền tố và MỌI số đoạn*. Ta không cần nhiều thế! Ta chỉ cần **một con số**. Liệu có câu hỏi nào **đơn giản hơn** mà vẫn dẫn tới con số đó?

## 9. Bước đột phá: đổi câu hỏi "bao nhiêu?" thành "được không?"

Câu hỏi gốc (khó): *"Đoạn lớn nhất **nhỏ nhất** có thể là bao nhiêu?"*

Câu hỏi mới (dễ hơn nhiều): *"Cho một con số `X`. Có cách chia nào thành `k` đoạn mà **mọi đoạn có tổng ≤ X** không?"* — gọi là `can(X)`, trả lời **Có / Không** (`True / False`).

Liên hệ đời thường: thay vì hỏi *"xe tải cần chở tối thiểu bao nhiêu kg?"*, ta hỏi *"nếu mỗi xe chở tối đa `X` kg thì có đủ `k` xe không?"*. Câu sau thì ai cũng trả lời được: **cứ chất hàng lên xe hiện tại cho tới khi chất thêm thùng tiếp theo sẽ quá `X`, lúc đó gọi xe mới**. Đếm xem cần bao nhiêu xe.

### Thuật toán tham lam (greedy) đếm số đoạn tối thiểu với giới hạn `X`

```text
pieces = 1, cur = 0                # đang mở đoạn thứ 1, tổng hiện tại 0
với mỗi x trong nums:
    nếu cur + x > X:   pieces += 1, cur = x     # đóng đoạn cũ, x mở đoạn mới
    ngược lại:         cur += x                 # x vào đoạn hiện tại
```

**Ví dụ số** với `[7,2,5,10,8]`:

- `X = 18`: `7 → 9 → 14` (thêm 10 thành 24 > 18, cắt) `| 10 → 18` → **2 đoạn** `[7,2,5] [10,8]`. 2 ≤ k=2 → `can(18) = True`.
- `X = 17`: `7 → 9 → 14` (cắt) `| 10` (thêm 8 thành 18 > 17, cắt) `| 8` → **3 đoạn**. 3 > 2 → `can(17) = False`.

Vậy `17` không làm được, `18` làm được → đáp án là `18`. Ta vừa giải bài bằng hai lần đếm `O(n)`!

**Lưu ý quan trọng:** nếu `X < max(nums)` thì phần tử lớn nhất **không lọt vào đoạn nào** — `can(X)` chắc chắn `False`. Hàm đếm phải tự phát hiện điều này (hoặc ta không bao giờ hỏi những `X` như vậy — xem mục 11).

In [13]:
def count_pieces(nums, X):
    """Số đoạn ít nhất để mọi đoạn có tổng <= X (tham lam). Trả về inf nếu không thể."""
    pieces, cur = 1, 0
    for x in nums:
        if x > X:                    # một phần tử đã vượt X -> không có cách nào
            return float("inf")
        if cur + x > X:
            pieces += 1
            cur = x
        else:
            cur += x
    return pieces


def greedy_groups(nums, X):
    """Giống count_pieces nhưng trả về chính các đoạn (để in/trace). Giả sử X >= max(nums)."""
    groups, cur = [[]], 0
    for x in nums:
        if cur + x > X:
            groups.append([])
            cur = 0
        groups[-1].append(x)
        cur += x
    return groups


nums = [7, 2, 5, 10, 8]
for X in [9, 10, 14, 17, 18, 32]:
    p = count_pieces(nums, X)
    print(f"X={X:>2}: count_pieces={p}", "" if p == float("inf") else f"-> {greedy_groups(nums, X)}")

X= 9: count_pieces=inf 
X=10: count_pieces=4 -> [[7, 2], [5], [10], [8]]
X=14: count_pieces=3 -> [[7, 2, 5], [10], [8]]
X=17: count_pieces=3 -> [[7, 2, 5], [10], [8]]
X=18: count_pieces=2 -> [[7, 2, 5], [10, 8]]
X=32: count_pieces=1 -> [[7, 2, 5, 10, 8]]


### Trace greedy: từng phần tử, quyết định "vào đoạn cũ" hay "mở đoạn mới"

Hàm `trace_greedy` in trạng thái **trước** khi quyết định (`cur` trước, `cur + x`), so với `X`, rồi hành động. So sánh hai giới hạn `X = 17` và `X = 18` trên Example 1.

In [14]:
def trace_greedy(nums, X):
    pieces, cur = 1, 0
    print(f"--- X = {X} ---")
    for i, x in enumerate(nums):
        if cur + x > X:
            print(f"i={i} x={x:>2}: cur={cur:>2}, cur+x={cur + x:>2} > {X} -> đóng đoạn {pieces}, mở đoạn {pieces + 1} với cur={x}")
            pieces += 1
            cur = x
        else:
            print(f"i={i} x={x:>2}: cur={cur:>2}, cur+x={cur + x:>2} <= {X} -> thêm vào đoạn {pieces}, cur={cur + x}")
            cur += x
    print(f"=> cần {pieces} đoạn")
    return pieces


trace_greedy([7, 2, 5, 10, 8], 17)
trace_greedy([7, 2, 5, 10, 8], 18)

--- X = 17 ---
i=0 x= 7: cur= 0, cur+x= 7 <= 17 -> thêm vào đoạn 1, cur=7
i=1 x= 2: cur= 7, cur+x= 9 <= 17 -> thêm vào đoạn 1, cur=9
i=2 x= 5: cur= 9, cur+x=14 <= 17 -> thêm vào đoạn 1, cur=14
i=3 x=10: cur=14, cur+x=24 > 17 -> đóng đoạn 1, mở đoạn 2 với cur=10
i=4 x= 8: cur=10, cur+x=18 > 17 -> đóng đoạn 2, mở đoạn 3 với cur=8
=> cần 3 đoạn
--- X = 18 ---
i=0 x= 7: cur= 0, cur+x= 7 <= 18 -> thêm vào đoạn 1, cur=7
i=1 x= 2: cur= 7, cur+x= 9 <= 18 -> thêm vào đoạn 1, cur=9
i=2 x= 5: cur= 9, cur+x=14 <= 18 -> thêm vào đoạn 1, cur=14
i=3 x=10: cur=14, cur+x=24 > 18 -> đóng đoạn 1, mở đoạn 2 với cur=10
i=4 x= 8: cur=10, cur+x=18 <= 18 -> thêm vào đoạn 2, cur=18
=> cần 2 đoạn


2

### Vì sao greedy cho **số đoạn ít nhất**? (lập luận "luôn đi trước")

Xét một cách chia hợp lệ **bất kỳ** (mọi đoạn ≤ X), gọi là "cách B". Gọi `e_t` là vị trí kết thúc đoạn thứ `t` của greedy, `b_t` là vị trí kết thúc đoạn thứ `t` của cách B. Ta chứng minh **`e_t >= b_t` với mọi `t`** (greedy luôn "đi trước" hoặc ngang hàng):

- `t = 1`: greedy kéo đoạn đầu **dài nhất có thể** mà tổng vẫn ≤ X. Đoạn đầu của B cũng có tổng ≤ X, nên không thể dài hơn đoạn đầu của greedy → `e_1 >= b_1`.
- Từ `t` sang `t + 1`: đoạn thứ `t+1` của B là `nums[b_t : b_{t+1}]`, tổng ≤ X. Greedy bắt đầu đoạn `t+1` tại `e_t >= b_t`, tức là **muộn hơn hoặc bằng**. Phần `nums[e_t : b_{t+1}]` là một khúc con của đoạn đó của B, nên tổng **không lớn hơn** (vì mọi số ≥ 0) → vẫn ≤ X → greedy kéo được ít nhất tới `b_{t+1}`.

Nếu B dùng `q` đoạn thì `b_q = n`, kéo theo `e_q = n`: greedy đã phủ hết mảng trong **không quá `q` đoạn**. Vì B là bất kỳ, greedy cho **số đoạn ít nhất**.

Ví dụ: với `X = 17`, cách chia `[7,2] [5,10] [8]` (tổng 9, 15, 8) cũng hợp lệ và dùng 3 đoạn; greedy `[7,2,5] [10] [8]` cũng 3 đoạn — không thể ít hơn.

### "≤ k đoạn" hay "đúng k đoạn"?

Greedy cho số đoạn **ít nhất** `p`. Đề đòi **đúng** `k`. Nếu `p < k` thì sao? Vì `k <= n`, luôn có một đoạn dài ≥ 2 phần tử khi số đoạn còn < `k`. **Tách** nó làm đôi: hai phần có tổng ≤ tổng đoạn cũ ≤ X (nhờ `nums[i] >= 0`!). Lặp lại tới khi đủ `k` đoạn. Vậy:

> **`can(X)` = `count_pieces(nums, X) <= k`** (dấu **≤**, không phải `==`).

Ví dụ `[1,2,3,4,5]`, `k = 2`, `X = 15`: greedy chỉ cần 1 đoạn. Tách thành `[1] [2,3,4,5]` → tổng 1 và 14, đều ≤ 15. Hợp lệ với đúng 2 đoạn.

## 10. Tính đơn điệu (monotonicity) — chìa khóa cho binary search

Nếu chia được với giới hạn `X` thì chắc chắn chia được với giới hạn `X + 1` (giữ nguyên cách chia cũ!). Nói cách khác, khi `X` tăng, `count_pieces` **không bao giờ tăng**, và `can(X)` có dạng:

```text
X      : ... 14     15     16     17     18    19    20 ...
can(X) : ... False  False  False  False  True  True  True ...
```

Một dãy `False ... False True ... True`. **Đáp án = vị trí `True` đầu tiên**. Cell dưới in toàn bộ dãy cho Example 1 với `X` từ `max(nums) = 10` tới `sum(nums) = 32`.

In [15]:
nums, k = [7, 2, 5, 10, 8], 2
row_x, row_p, row_c = [], [], []
for X in range(max(nums), sum(nums) + 1):
    p = count_pieces(nums, X)
    row_x.append(f"{X:>3}")
    row_p.append(f"{p:>3}")
    row_c.append("  T" if p <= k else "  F")
print("X      :", "".join(row_x))
print("pieces :", "".join(row_p))
print("can(X) :", "".join(row_c))

X      :  10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32
pieces :   4  4  4  4  3  3  3  3  2  2  2  2  2  2  2  2  2  2  2  2  2  2  1
can(X) :   F  F  F  F  F  F  F  F  T  T  T  T  T  T  T  T  T  T  T  T  T  T  T


Quan sát: `pieces` đi `4 4 4 4 3 3 3 3 2 2 ... 2 1` — **không tăng**; `can(X)` đổi từ `F` sang `T` **đúng một lần**, tại `X = 18`.

### Bước trung gian (cách 4a): thử lần lượt từng `X`

Có tính đơn điệu rồi, cách đơn giản nhất: thử `X = max(nums), max(nums)+1, ...` cho tới `True` đầu tiên. Đúng, nhưng có thể phải thử tới `sum - max` giá trị — với `sum` tới `10^9` thì hàng tỷ lần gọi `count_pieces`. Quá chậm. Ta cần cách **nhảy** tới chỗ đổi `F → T` nhanh hơn: **binary search**.

In [16]:
def split_array_try_all_X(nums, k):
    X = max(nums)
    tries = 1
    while count_pieces(nums, X) > k:
        X += 1
        tries += 1
    return X, tries


for nums, k, expected in examples:
    ans, tries = split_array_try_all_X(nums, k)
    print(nums, k, "->", ans, "| mong đợi", expected, "| số X đã thử:", tries)

big = [10**6] * 1000                     # mảng lớn nhất theo constraints
print(f"n=1000, mọi phần tử 10^6, k=1: số X sẽ phải thử = {sum(big) - max(big) + 1:,}")

[7, 2, 5, 10, 8] 2 -> 18 | mong đợi 18 | số X đã thử: 9
[1, 2, 3, 4, 5] 2 -> 9 | mong đợi 9 | số X đã thử: 5
n=1000, mọi phần tử 10^6, k=1: số X sẽ phải thử = 999,000,001


## 11. Nền tảng 4: binary search (tìm kiếm nhị phân) từ căn bản

**Trò chơi đoán số:** bạn tôi nghĩ một số từ 1 tới 100, mỗi lần tôi đoán thì bạn nói "lớn hơn" hay "nhỏ hơn". Đoán `1, 2, 3, ...` có thể mất 100 lần. Chiến thuật thông minh: luôn đoán **chính giữa** khoảng còn lại → mỗi lần loại **một nửa** → tối đa khoảng `log2(100) ≈ 7` lần.

Binary search áp dụng được khi có **tính đơn điệu**: nhìn một vị trí `mid` là biết đáp án nằm bên trái hay bên phải.

### Mẫu "tìm `True` đầu tiên" (rất nên thuộc)

Cho dãy dạng `F F F T T T` (biết chắc vị trí `hi` ban đầu là `T`). Tìm chỉ số `True` đầu tiên:

```python
lo, hi = <đầu>, <cuối>          # đáp án luôn nằm trong [lo, hi]
while lo < hi:                  # còn hơn 1 ứng viên
    mid = (lo + hi) // 2
    if ok(mid):  hi = mid       # mid là True: đáp án <= mid (mid có thể chính là đáp án, GIỮ nó)
    else:        lo = mid + 1   # mid là False: đáp án > mid (BỎ mid)
return lo                       # lo == hi
```

Ba chi tiết đi chung với nhau:

1. `while lo < hi` (không có `=`): dừng khi khoảng còn đúng 1 phần tử — đó là đáp án.
2. `mid = (lo + hi) // 2` làm tròn **xuống** → luôn có `lo <= mid < hi`.
3. Vì `mid < hi`, phép `hi = mid` làm khoảng **thực sự co lại**; `lo = mid + 1` cũng vậy → vòng lặp chắc chắn dừng.

In [17]:
def first_true(ok, lo, hi):
    """Trả về x nhỏ nhất trong [lo, hi] mà ok(x) True (giả sử ok(hi) True và ok đơn điệu)."""
    steps = 0
    while lo < hi:
        steps += 1
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo, steps


secret = 73
print("đoán số 1..100, số bí mật 73:", first_true(lambda x: x >= secret, 1, 100))

flags = [False, False, False, True, True, True, True]
print("vị trí True đầu tiên trong", flags, ":", first_true(lambda i: flags[i], 0, len(flags) - 1))

print("1..10^9, ngưỡng 123456789:", first_true(lambda x: x >= 123456789, 1, 10**9))

đoán số 1..100, số bí mật 73: (73, 7)
vị trí True đầu tiên trong [False, False, False, True, True, True, True] : (3, 3)
1..10^9, ngưỡng 123456789: (123456789, 30)


Chỉ khoảng **30 bước** cho khoảng `1..10^9` (vì `2^30 ≈ 1.07 · 10^9`). Đó chính là thứ ta cần để thay cho "hàng tỷ lần thử" ở cách 4a.

Để ý: hàm `first_true` **không cần mảng** — nó chỉ cần một hàm `ok(x)` đơn điệu trên một **khoảng số nguyên**. Đó là ý nghĩa của "binary search **trên đáp án**": ta không tìm trong `nums`, ta tìm trong **khoảng các giá trị đáp án có thể có**.

## 12. Chọn khoảng tìm kiếm: vì sao `lo = max(nums)`, `hi = sum(nums)`?

Ta cần một khoảng `[lo, hi]` **chắc chắn chứa đáp án**:

- **Cận dưới `lo = max(nums)`:** phần tử lớn nhất phải nằm trong một đoạn nào đó, nên đoạn đó có tổng ≥ `max(nums)` → đáp án ≥ `max(nums)`. Và đây là đáp án thật khi `k = n` (mỗi phần tử một đoạn). Bonus: mọi `X` ta hỏi đều ≥ `max(nums)`, nên không bao giờ rơi vào tình huống "một phần tử vượt X".
- **Cận trên `hi = sum(nums)`:** mọi đoạn có tổng ≤ tổng cả mảng → đáp án ≤ `sum(nums)`. Và `can(sum) = True` luôn đúng (1 đoạn ≤ k). Đây là đáp án thật khi `k = 1`.

Ví dụ Example 1: `lo = 10`, `hi = 32`, đáp án `18` nằm giữa. Có thể chọn `lo = 0` hay `hi = 10^9` cũng "đúng" về mặt khoảng, nhưng `lo = 0` cực kỳ nguy hiểm nếu hàm đếm không kiểm tra `x > X` (xem Lỗi 1 ở mục 18).

In [18]:
for nums, k in [([7, 2, 5, 10, 8], 2), ([7, 2, 5, 10, 8], 5), ([7, 2, 5, 10, 8], 1), ([0, 0, 0], 2)]:
    print(f"nums={nums}, k={k}: lo=max={max(nums)}, hi=sum={sum(nums)}, đáp án (brute)={split_array_bruteforce(nums, k)}")

nums=[7, 2, 5, 10, 8], k=2: lo=max=10, hi=sum=32, đáp án (brute)=18
nums=[7, 2, 5, 10, 8], k=5: lo=max=10, hi=sum=32, đáp án (brute)=10
nums=[7, 2, 5, 10, 8], k=1: lo=max=10, hi=sum=32, đáp án (brute)=32
nums=[0, 0, 0], k=2: lo=max=0, hi=sum=0, đáp án (brute)=0


## 13. Cách 4 (cách chính) — Binary search trên đáp án + greedy kiểm tra

Ghép mọi mảnh lại:

1. **Khoảng đáp án:** `lo = max(nums)`, `hi = sum(nums)` (mục 12).
2. **Câu hỏi Có/Không:** `can(X) = count_pieces(nums, X) <= k` (mục 9), đơn điệu `F...F T...T` (mục 10).
3. **Tìm `True` đầu tiên** bằng mẫu `while lo < hi` (mục 11):
   - `can(mid)` đúng → đáp án ≤ `mid` → `hi = mid`.
   - `can(mid)` sai → đáp án > `mid` → `lo = mid + 1`.
4. Khi `lo == hi`, đó là đáp án.

Ta viết hàm đếm **ngay bên trong** cho gọn và không cần kiểm tra `x > X` (vì mọi `mid >= lo = max(nums)`). Ngoài ra, có thể **dừng sớm** khi số đoạn đã vượt `k` — không cần đếm tiếp.

In [19]:
def split_array_binary_search(nums, k):
    def can(X):
        pieces, cur = 1, 0
        for x in nums:
            if cur + x > X:
                pieces += 1
                cur = x
                if pieces > k:          # dừng sớm: đã chắc chắn không được
                    return False
            else:
                cur += x
        return True

    lo, hi = max(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if can(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


for nums, k, expected in examples:
    print(nums, k, "->", split_array_binary_search(nums, k), "| mong đợi", expected)
for nums, k in [([7, 2, 5, 10, 8], 3), ([1, 4, 4], 3), ([0, 0, 0], 2), ([5], 1)]:
    print(nums, k, "->", split_array_binary_search(nums, k), "| brute:", split_array_bruteforce(nums, k))

[7, 2, 5, 10, 8] 2 -> 18 | mong đợi 18
[1, 2, 3, 4, 5] 2 -> 9 | mong đợi 9
[7, 2, 5, 10, 8] 3 -> 14 | brute: 14
[1, 4, 4] 3 -> 4 | brute: 4
[0, 0, 0] 2 -> 0 | brute: 0
[5] 1 -> 5 | brute: 5


## 14. Trace cách 4 trên Example 1

`nums = [7,2,5,10,8]`, `k = 2`, `lo = 10`, `hi = 32`. Mỗi bước: tính `mid`, chia tham lam với giới hạn `mid`, đếm số đoạn, so với `k`, rồi thu hẹp khoảng.

| bước | lo | hi | mid | chia tham lam với giới hạn mid | số đoạn | ≤ k? | cập nhật |
|---:|---:|---:|---:|---|---:|---|---|
| 1 | 10 | 32 | 21 | `[7,2,5] [10,8]` | 2 | có | `hi = 21` |
| 2 | 10 | 21 | 15 | `[7,2,5] [10] [8]` | 3 | không | `lo = 16` |
| 3 | 16 | 21 | 18 | `[7,2,5] [10,8]` | 2 | có | `hi = 18` |
| 4 | 16 | 18 | 17 | `[7,2,5] [10] [8]` | 3 | không | `lo = 18` |
| — | 18 | 18 | | `lo == hi` → dừng | | | đáp án **18** |

Chỉ **4 lần** gọi `can` thay vì thử 9 giá trị `10..18` như cách 4a. Ý nghĩa invariant: ở mọi bước, đáp án luôn nằm trong `[lo, hi]` — `hi` luôn là một giá trị **đã biết là làm được** (hoặc `sum`), `lo - 1` luôn là giá trị **đã biết là không làm được** (hoặc `< max`).

In [20]:
def trace_split_array_bs(nums, k):
    lo, hi = max(nums), sum(nums)
    print(f"nums={nums}, k={k}: lo=max={lo}, hi=sum={hi}")
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        groups = greedy_groups(nums, mid)
        ok = len(groups) <= k
        action = f"hi = {mid}" if ok else f"lo = {mid + 1}"
        print(f"bước {step}: lo={lo:>2} hi={hi:>2} mid={mid:>2} | {groups} -> {len(groups)} đoạn "
              f"{'<=' if ok else '> '} k -> {action}")
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print(f"lo == hi == {lo} -> đáp án {lo}\n")
    return lo


trace_split_array_bs([7, 2, 5, 10, 8], 2)
trace_split_array_bs([1, 2, 3, 4, 5], 2)

nums=[7, 2, 5, 10, 8], k=2: lo=max=10, hi=sum=32
bước 1: lo=10 hi=32 mid=21 | [[7, 2, 5], [10, 8]] -> 2 đoạn <= k -> hi = 21
bước 2: lo=10 hi=21 mid=15 | [[7, 2, 5], [10], [8]] -> 3 đoạn >  k -> lo = 16
bước 3: lo=16 hi=21 mid=18 | [[7, 2, 5], [10, 8]] -> 2 đoạn <= k -> hi = 18
bước 4: lo=16 hi=18 mid=17 | [[7, 2, 5], [10], [8]] -> 3 đoạn >  k -> lo = 18
lo == hi == 18 -> đáp án 18

nums=[1, 2, 3, 4, 5], k=2: lo=max=5, hi=sum=15
bước 1: lo= 5 hi=15 mid=10 | [[1, 2, 3, 4], [5]] -> 2 đoạn <= k -> hi = 10
bước 2: lo= 5 hi=10 mid= 7 | [[1, 2, 3], [4], [5]] -> 3 đoạn >  k -> lo = 8
bước 3: lo= 8 hi=10 mid= 9 | [[1, 2, 3], [4, 5]] -> 2 đoạn <= k -> hi = 9
bước 4: lo= 8 hi= 9 mid= 8 | [[1, 2, 3], [4], [5]] -> 3 đoạn >  k -> lo = 9
lo == hi == 9 -> đáp án 9



9

Trace thứ hai (Example 2, `lo = 5`, `hi = 15`) cho thấy một điều thú vị ở bước cuối: khi `lo = 8, hi = 9`, `mid = 8` (làm tròn xuống) → `8` không làm được → `lo = 9 = hi`, dừng. Nhờ làm tròn xuống mà `mid` không bao giờ bằng `hi`, khoảng luôn co lại.

Thêm một trace có **số 0**: `[0,3,0,1,2]`, `k = 3`. Để ý hai điều: (1) số 0 "dính" vào đoạn nào cũng không làm tăng tổng; (2) ở bước 2, greedy chỉ cần **2 đoạn** (`< k = 3`) mà vẫn được tính là làm được — đúng như lập luận "≤ k là đủ" ở mục 9: tách `[0,3,0] [1,2]` thành `[0,3,0] [1] [2]`, max vẫn là 3. Đáp án bằng đúng cận dưới `lo = max(nums) = 3`.

In [21]:
trace_split_array_bs([0, 3, 0, 1, 2], 3)

nums=[0, 3, 0, 1, 2], k=3: lo=max=3, hi=sum=6
bước 1: lo= 3 hi= 6 mid= 4 | [[0, 3, 0, 1], [2]] -> 2 đoạn <= k -> hi = 4
bước 2: lo= 3 hi= 4 mid= 3 | [[0, 3, 0], [1, 2]] -> 2 đoạn <= k -> hi = 3
lo == hi == 3 -> đáp án 3



3

## 15. Code nộp LeetCode

Chuyển cách 4 vào đúng khuôn `class Solution`:

- Tên method `splitArray`, tham số `self, nums: List[int], k: int`, trả về `int`.
- `List` đã được LeetCode import sẵn (trong notebook ta đã `from typing import List` ở đầu).
- Không in gì trong code nộp. Hàm kiểm tra viết thành method phụ `_can` cho dễ đọc.

In [22]:
class Solution:
    def splitArray(self, nums: List[int], k: int) -> int:
        lo, hi = max(nums), sum(nums)          # đáp án luôn nằm trong [lo, hi]
        while lo < hi:
            mid = (lo + hi) // 2
            if self._can(nums, k, mid):        # chia được với max <= mid
                hi = mid
            else:
                lo = mid + 1
        return lo

    def _can(self, nums: List[int], k: int, X: int) -> bool:
        pieces, cur = 1, 0
        for x in nums:
            if cur + x > X:                    # x không vừa đoạn hiện tại -> mở đoạn mới
                pieces += 1
                cur = x
                if pieces > k:
                    return False
            else:
                cur += x
        return True


sol = Solution()
# Ví dụ của đề
assert sol.splitArray([7, 2, 5, 10, 8], 2) == 18
assert sol.splitArray([1, 2, 3, 4, 5], 2) == 9
# Edge case
assert sol.splitArray([5], 1) == 5                        # n = 1
assert sol.splitArray([1, 2, 3, 4, 5], 1) == 15           # k = 1 -> sum
assert sol.splitArray([1, 2, 3, 4, 5], 5) == 5            # k = n -> max
assert sol.splitArray([0, 0, 0], 2) == 0                  # toàn số 0
assert sol.splitArray([1, 4, 4], 3) == 4                  # bẫy "== k"
assert sol.splitArray([10, 1, 1], 3) == 10
assert sol.splitArray([2, 3, 1, 2, 4, 3], 3) == 6
assert sol.splitArray([10**6] * 1000, 50) == 20 * 10**6   # giới hạn lớn nhất
print("class Solution PASS toàn bộ test!")

class Solution PASS toàn bộ test!


### Kiểm thử đối chiếu chéo 4 cách

Sinh ngẫu nhiên mảng nhỏ (để brute force chạy nổi), giá trị trong `0..20` (cố ý có số 0 và số trùng), `k` ngẫu nhiên trong `1..n` — đúng như constraints. `random.seed(410)` cố định để mỗi lần chạy ra cùng bộ test. Sau đó so riêng DP với binary search trên mảng dài hơn (brute force không chạy nổi nữa).

In [23]:
import random

random.seed(410)
methods = [split_array_memo, split_array_dp, split_array_binary_search, Solution().splitArray]
for _ in range(1500):
    n = random.randint(1, 9)
    nums = [random.randint(0, 20) for _ in range(n)]
    k = random.randint(1, n)
    expected = split_array_bruteforce(nums, k)
    for f in methods:
        assert f(nums, k) == expected, (f.__name__, nums, k)
print("4 cách + class Solution khớp nhau trên 1500 input nhỏ ngẫu nhiên")

for _ in range(100):
    n = random.randint(1, 60)
    nums = [random.randint(0, 10**6) for _ in range(n)]
    k = random.randint(1, min(50, n))
    assert split_array_dp(nums, k) == Solution().splitArray(nums, k), (nums, k)
print("DP và binary search khớp nhau trên 100 input dài tới 60 phần tử, giá trị tới 10^6")

4 cách + class Solution khớp nhau trên 1500 input nhỏ ngẫu nhiên


DP và binary search khớp nhau trên 100 input dài tới 60 phần tử, giá trị tới 10^6


## 16. Vì sao cách 4 đúng? (invariant bằng lời)

Ba mệnh đề, mỗi cái đã được giải thích ở trên:

1. **Greedy đúng** (mục 9): `count_pieces(nums, X)` là số đoạn **ít nhất** để mọi đoạn ≤ X — nhờ lập luận "greedy luôn đi trước" và `nums[i] >= 0`.
2. **≤ k là đủ** (mục 9): nếu chia được thành `p <= k` đoạn thì tách tiếp được thành đúng `k` đoạn mà max không tăng (vì `k <= n` và số không âm). Vậy `can(X)` ⇔ "tồn tại cách chia **đúng** `k` đoạn với max ≤ X".
3. **Đơn điệu** (mục 10): `can(X)` ⇒ `can(X + 1)`. Đáp án bài = `X` nhỏ nhất có `can(X) = True`.

**Invariant của vòng lặp:** *đáp án luôn nằm trong `[lo, hi]`*.

- Ban đầu đúng: đáp án ∈ `[max(nums), sum(nums)]` (mục 12).
- Nếu `can(mid)` đúng: đáp án ≤ `mid`, gán `hi = mid` vẫn giữ đáp án trong khoảng.
- Nếu `can(mid)` sai: theo đơn điệu mọi `X ≤ mid` đều sai, đáp án ≥ `mid + 1`, gán `lo = mid + 1` vẫn đúng.
- Mỗi vòng khoảng co lại ít nhất 1 (vì `lo <= mid < hi`) → vòng lặp dừng khi `lo == hi`, và khoảng chỉ còn một giá trị: chính là đáp án.

## 17. Độ phức tạp và so sánh các cách

Gọi `n = len(nums)`, `S = sum(nums)`, `M = max(nums)`.

- **Cách 4:** số vòng binary search = `⌈log2(S - M + 1)⌉`. Với `S <= 10^9` thì tối đa khoảng **30 vòng**. Mỗi vòng `can` duyệt mảng `O(n)`. → **Time `O(n · log S)`** ≈ `30 · 1000 = 3 · 10^4` phép tính ở giới hạn đề. **Space `O(1)`** (chỉ vài biến). Không phụ thuộc `k`!

| Cách | Ý tưởng | Time | Space | Ở giới hạn đề (n=1000, k=50) |
|---|---|---|---|---|
| 1. Brute force | thử mọi tập `k-1` vách | `O(C(n-1,k-1) · k)` | `O(n)` | ~`10^83` — không thể |
| 2. Đệ quy + memo | `f(i, m)` trên phần đuôi | `O(k · n²)` | `O(k · n)` | ~`5·10^7` — chậm trong Python |
| 3. DP bảng | `dp[m][j]` trên tiền tố | `O(k · n²)` | `O(k · n)` (giảm được `O(n)`) | ~`5·10^7` — chậm trong Python |
| 4a. Thử từng X | đơn điệu + quét tuyến tính | `O(n · (S - M))` | `O(1)` | tới ~`10^12` — không thể |
| **4. Binary search + greedy** | đơn điệu + chia đôi khoảng | **`O(n · log S)`** | **`O(1)`** | **~`3·10^4` — tức thì** |

Cell dưới đo thời gian thật (con số cụ thể tùy máy) và đếm số lần gọi `can` trên một input kích thước tối đa.

In [24]:
import time

random.seed(7)
nums = [random.randint(0, 10**6) for _ in range(300)]
k = 15
t0 = time.perf_counter(); a = split_array_dp(nums, k); t1 = time.perf_counter()
b = split_array_binary_search(nums, k); t2 = time.perf_counter()
assert a == b
print(f"n=300, k=15: DP {t1 - t0:.3f}s | binary search {t2 - t1:.5f}s | đáp án {a}")

calls = 0
def can_counted(nums, k, X):
    global calls
    calls += 1
    return Solution()._can(nums, k, X)

nums = [random.randint(0, 10**6) for _ in range(1000)]
k = 50
lo, hi = max(nums), sum(nums)
print(f"n=1000, k=50: khoảng [lo, hi] dài {hi - lo + 1:,} giá trị, log2 ≈ {math.log2(hi - lo + 1):.1f}")
while lo < hi:
    mid = (lo + hi) // 2
    if can_counted(nums, k, mid):
        hi = mid
    else:
        lo = mid + 1
assert lo == Solution().splitArray(nums, k)
print("đáp án:", lo, "| số lần gọi can:", calls)

n=300, k=15: DP 0.055s | binary search 0.00051s | đáp án 9654088
n=1000, k=50: khoảng [lo, hi] dài 491,865,578 giá trị, log2 ≈ 28.9
đáp án: 10225291 | số lần gọi can: 29


## 18. Lỗi người mới hay gặp

### Lỗi 1: `lo = 0` (hoặc `lo = 1`) mà hàm đếm không kiểm tra phần tử vượt X

Nhiều bạn nghĩ "cận dưới cứ cho 0 cho an toàn". Nhưng hàm đếm greedy "trơn" (không có dòng `if x > X`) khi gặp một phần tử lớn hơn `X` vẫn vui vẻ "mở đoạn mới" chứa phần tử đó — một đoạn **vượt giới hạn** — và báo số đoạn nhỏ giả tạo. Với `[1,4,4]`, `k = 3`, `X = 1`: hàm đếm ra `[1] [4] [4]` = 3 đoạn ≤ k → tưởng là được → trả về **1**, trong khi đáp án đúng là **4**.

**Sửa:** dùng `lo = max(nums)` (khuyên dùng), hoặc thêm `if x > X: return False` trong hàm kiểm tra.

### Lỗi 2: Dùng `== k` thay vì `<= k`

Nghĩ rằng "đề đòi đúng k đoạn nên kiểm tra `pieces == k`". Nhưng greedy cho số đoạn **ít nhất**; `pieces < k` vẫn là **làm được** (tách thêm). Dùng `==` làm hỏng tính đơn điệu: với `[1,4,4]`, `k = 3`, mọi `X >= 5` cho `pieces <= 2` nên bị coi là "không được" → binary search chạy thẳng lên `hi = sum = 9`.

In [25]:
def count_pieces_naive(nums, X):          # KHÔNG kiểm tra x > X
    pieces, cur = 1, 0
    for x in nums:
        if cur + x > X:
            pieces += 1
            cur = x
        else:
            cur += x
    return pieces


def bs_generic(nums, k, lo, ok):
    hi = sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


nums, k = [1, 4, 4], 3
print("Đáp án đúng            :", split_array_bruteforce(nums, k))
print("Lỗi 1 (lo=0, đếm trơn) :", bs_generic(nums, k, 0, lambda X: count_pieces_naive(nums, X) <= k))
print("Lỗi 2 (== k)           :", bs_generic(nums, k, max(nums), lambda X: count_pieces(nums, X) == k))
print("Đúng (lo=max, <= k)    :", bs_generic(nums, k, max(nums), lambda X: count_pieces(nums, X) <= k))

Đáp án đúng            : 4
Lỗi 1 (lo=0, đếm trơn) : 1
Lỗi 2 (== k)           : 9
Đúng (lo=max, <= k)    : 4


### Lỗi 3: `while lo <= hi` đi kèm `hi = mid` → lặp vô hạn

Mẫu "`lo <= hi`" thuộc về kiểu binary search **tìm đúng một giá trị** (`hi = mid - 1`, `lo = mid + 1`). Trộn nó với `hi = mid`: khi `lo == hi == mid` và `can(mid)` đúng thì `hi = mid` không đổi gì → vòng lặp quay mãi. Cell dưới dùng "cầu chì" giới hạn 100 vòng để minh họa mà không treo notebook.

### Lỗi 4: Bắt đầu đếm `pieces = 0`, hoặc khi cắt lại gán `cur = 0`

- `pieces = 0`: quên tính **đoạn cuối cùng** (đoạn đang mở khi hết mảng). Số đoạn bị đếm thiếu 1 → tưởng làm được trong khi thật ra không → đáp án **nhỏ hơn** đúng. Example 1 ra `14` (chính là đáp án của `k = 3`).
- Khi cắt gán `cur = 0` thay vì `cur = x`: phần tử `x` hiện tại bị **rơi mất**, không thuộc đoạn nào.

In [26]:
def bs_lo_le_hi_buggy(nums, k):
    lo, hi = max(nums), sum(nums)
    rounds = 0
    while lo <= hi:                          # SAI khi đi với hi = mid
        rounds += 1
        if rounds > 100:
            return f"lặp vô hạn (dừng ở lo={lo}, hi={hi})"
        mid = (lo + hi) // 2
        if count_pieces(nums, mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


def count_pieces_start_zero(nums, X):        # SAI: quên đoạn cuối
    pieces, cur = 0, 0
    for x in nums:
        if cur + x > X:
            pieces += 1
            cur = x
        else:
            cur += x
    return pieces


def count_pieces_drop_x(nums, X):            # SAI: cur = 0 làm rơi mất x
    pieces, cur = 1, 0
    for x in nums:
        if cur + x > X:
            pieces += 1
            cur = 0
        else:
            cur += x
    return pieces


nums, k = [7, 2, 5, 10, 8], 2
print("Lỗi 3:", bs_lo_le_hi_buggy(nums, k))
print("Lỗi 4a (pieces=0):", bs_generic(nums, k, max(nums), lambda X: count_pieces_start_zero(nums, X) <= k), "| đúng: 18")
print("Lỗi 4b (cur=0)   :", bs_generic(nums, k, max(nums), lambda X: count_pieces_drop_x(nums, X) <= k), "| đúng: 18")

Lỗi 3: lặp vô hạn (dừng ở lo=18, hi=18)
Lỗi 4a (pieces=0): 14 | đúng: 18
Lỗi 4b (cur=0)   : 14 | đúng: 18


### Lỗi 5: Nhầm "subarray" với "subset" — sắp xếp mảng trước

Có bạn thấy bài "chia đều tổng" liền `nums.sort()` cho dễ chia. **Sai**: đoạn phải **liên tiếp theo thứ tự gốc**. Sắp xếp làm thay đổi bài toán. Ví dụ `[1, 10, 1, 10]`, `k = 2`: đáp án đúng `11` (`[1,10] [1,10]`); nếu sắp xếp thành `[1,1,10,10]` thì ra `12` (`[1,1,10] [10]`) — chẳng liên quan.

### Lỗi 6 (với cách 2/3): sai biên vòng `for`

Trong cách 2, `j` phải chạy tới `n - m + 1` **bao gồm** (nên `range(i + 1, n - m + 2)`); thiếu `+2` sẽ bỏ sót cách để phần còn lại đúng `m - 1` phần tử. Trong cách 3, `t` bắt đầu từ `m - 1` chứ không phải `0`. Test đối chiếu với brute force (như mục 15) bắt được loại lỗi này ngay.

In [27]:
nums, k = [1, 10, 1, 10], 2
print("đúng (giữ thứ tự)  :", Solution().splitArray(nums, k))
print("sai (sắp xếp trước):", Solution().splitArray(sorted(nums), k))

đúng (giữ thứ tự)  : 11
sai (sắp xếp trước): 12


## 19. Mẹo nhận dạng pattern "binary search trên đáp án"

Hãy nghĩ tới kỹ thuật này khi đề có những dấu hiệu:

1. Cụm từ **"minimize the maximum"** (nhỏ nhất của cái lớn nhất) hoặc **"maximize the minimum"** (lớn nhất của cái nhỏ nhất), "smallest capacity", "minimum speed", "minimum days"...
2. Đáp án là **một số nguyên** trong một khoảng rõ ràng (ở đây `[max, sum]`).
3. Câu hỏi ngược *"nếu đáp án là X thì có làm được không?"* **dễ hơn hẳn** và kiểm tra được bằng greedy `O(n)`.
4. Có **tính đơn điệu**: X làm được thì X lớn hơn (với "minimize") cũng làm được.

Công thức tư duy 3 bước: **(1) khoảng đáp án `[lo, hi]` → (2) hàm `can(X)` đơn điệu → (3) tìm `True` đầu tiên** (hoặc cuối cùng, với "maximize the minimum").

## 20. Biến thể và bài liên quan

- **LeetCode 1011 — Capacity To Ship Packages Within D Days:** *y hệt* bài 410 (trọng lượng = `nums`, số ngày = `k`).
- **LeetCode 875 — Koko Eating Bananas:** tìm tốc độ nhỏ nhất; `can(v)` = tổng `ceil(pile / v)` ≤ `h`. (Bài tập 5.)
- **LeetCode 1482 — Minimum Number of Days to Make m Bouquets**, **2064 — Minimized Maximum of Products Distributed to Any Store**: cùng khuôn "minimize".
- **LeetCode 1552 — Magnetic Force Between Two Balls**, **1231 — Divide Chocolate** (Premium): dạng ngược "maximize the minimum" → tìm `True` **cuối cùng**. (Bài tập 6.)
- **LeetCode 1335 — Minimum Difficulty of a Job Schedule:** chia đoạn liên tiếp nhưng hàm chi phí là *tổng các max* → không đơn điệu kiểu này, phải dùng DP như cách 3.
- **Mở rộng tự nhiên:** in ra **chính cách chia** đạt đáp án, với **đúng** `k` đoạn. Ý tưởng: chạy greedy với `X = đáp án`, nhưng **cắt bắt buộc** khi số phần tử còn lại vừa đủ cho mỗi đoạn còn thiếu một phần tử. Cell dưới cài đặt và kiểm tra với brute force.

In [28]:
def split_array_reconstruct(nums, k):
    X = split_array_binary_search(nums, k)
    n = len(nums)
    parts, cur = [[]], 0
    for i, x in enumerate(nums):
        remaining = n - i                     # số phần tử còn lại, tính cả x
        need = k - len(parts)                 # số đoạn còn phải mở thêm
        if parts[-1] and (cur + x > X or remaining == need):
            parts.append([])                  # cắt (vì vượt X, hoặc bắt buộc để đủ k đoạn)
            cur = 0
        parts[-1].append(x)
        cur += x
    return X, parts


for nums, k in [([7, 2, 5, 10, 8], 2), ([1, 2, 3, 4, 5], 4), ([0, 0, 5, 0], 3), ([9, 1, 1, 1], 3)]:
    print(nums, k, "->", split_array_reconstruct(nums, k))

random.seed(1011)
for _ in range(1000):
    n = random.randint(1, 9)
    nums = [random.randint(0, 9) for _ in range(n)]
    k = random.randint(1, n)
    X, parts = split_array_reconstruct(nums, k)
    assert len(parts) == k and all(parts) and sum(parts, []) == nums
    assert max(sum(p) for p in parts) == X == split_array_bruteforce(nums, k)
print("reconstruct: đúng k đoạn, không rỗng, ghép lại = nums, max = đáp án (1000 test)")

[7, 2, 5, 10, 8] 2 -> (18, [[7, 2, 5], [10, 8]])
[1, 2, 3, 4, 5] 4 -> (5, [[1, 2], [3], [4], [5]])
[0, 0, 5, 0] 3 -> (5, [[0, 0], [5], [0]])
[9, 1, 1, 1] 3 -> (9, [[9], [1, 1], [1]])


reconstruct: đúng k đoạn, không rỗng, ghép lại = nums, max = đáp án (1000 test)


## 21. Bài tập vận dụng

Sáu bài, từ dễ tới khó. Hãy **tự làm trước** rồi mới xem gợi ý (mục 22) và đáp án (mục 23, cuối notebook). Các cell TODO chỉ **định nghĩa** hàm; các dòng test đang được comment — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

Với `nums = [1, 4, 4]`, hãy đoán giá trị của:

- `count_pieces(nums, X)` (bản **có** kiểm tra `x > X`, mục 9) với `X = 3, 4, 5, 8, 9`.
- `count_pieces_naive(nums, 3)` (bản **không** kiểm tra, mục 18).

Giá trị nào trong đó là "nói dối"? Vì sao?

### Bài 2 — Tính tay một lượt binary search

`nums = [2, 3, 1, 2, 4, 3]`, `k = 3`. Lập bảng giống mục 14: các cột `lo, hi, mid`, cách chia tham lam, số đoạn, cập nhật. Đáp án cuối cùng là bao nhiêu? Cần bao nhiêu lần gọi `can`?

### Bài 3 — Điền TODO: hàm kiểm tra `can_split`

Viết `can_split(nums, k, X)` trả về `True` nếu chia được `nums` thành **không quá** `k` đoạn liên tiếp, mỗi đoạn tổng ≤ `X`. Hàm phải tự trả `False` khi có phần tử lớn hơn `X` (để dùng được cả khi `lo = 0`).

In [29]:
def can_split(nums, k, X):
    # TODO 1: khởi tạo pieces = 1 (đoạn đang mở), cur = 0
    # TODO 2: duyệt từng x trong nums
    # TODO 3: nếu x > X -> return False
    # TODO 4: nếu cur + x > X -> mở đoạn mới (pieces += 1, cur = x); nếu pieces > k -> return False
    # TODO 5: ngược lại cur += x
    # TODO 6: hết vòng -> return True
    pass


# Tự làm xong rồi mới mở test:
# assert can_split([7, 2, 5, 10, 8], 2, 18) is True
# assert can_split([7, 2, 5, 10, 8], 2, 17) is False
# assert can_split([1, 4, 4], 3, 3) is False
# assert can_split([0, 0, 0], 1, 0) is True

### Bài 4 — Tự viết lại cách 4 từ đầu

Không nhìn lên trên, viết `my_split_array(nums, k)`: chọn `lo`, `hi`, vòng `while`, cập nhật. Có thể dùng `can_split` của Bài 3. Sau đó tự viết một vòng test ngẫu nhiên so với `split_array_bruteforce`.

In [30]:
def my_split_array(nums, k):
    # TODO: lo = ?, hi = ?
    # TODO: while lo < hi: mid = ?; nếu chia được với mid thì ... ngược lại ...
    # TODO: return ?
    raise NotImplementedError


# assert my_split_array([7, 2, 5, 10, 8], 2) == 18
# assert my_split_array([1, 2, 3, 4, 5], 2) == 9
# assert my_split_array([1, 4, 4], 3) == 4

### Bài 5 — Biến thể: LeetCode 875 — Koko Eating Bananas

Koko có `piles[i]` quả chuối ở đống thứ `i` và `h` giờ. Mỗi giờ Koko chọn **một** đống và ăn tối đa `v` quả từ đống đó (đống còn ít hơn `v` thì ăn hết đống và nghỉ hết giờ đó). Tìm **tốc độ `v` nhỏ nhất** (số nguyên) để ăn hết trong `h` giờ. Constraints của đề gốc: `1 <= piles.length <= 10^4`, `piles.length <= h <= 10^9`, `1 <= piles[i] <= 10^9`.

Ví dụ: `piles = [3,6,7,11], h = 8` → `4`; `piles = [30,11,23,4,20], h = 5` → `30`; `h = 6` → `23`.

Câu hỏi gợi mở: khoảng đáp án là gì? `can(v)` tính thế nào?

In [31]:
def min_eating_speed(piles, h):
    # TODO 1: lo = ?, hi = ?   (tốc độ nhỏ nhất có nghĩa là bao nhiêu? tốc độ nào chắc chắn đủ?)
    # TODO 2: hàm hours(v) = tổng số giờ để ăn hết với tốc độ v
    # TODO 3: binary search tìm v nhỏ nhất mà hours(v) <= h
    pass


# assert min_eating_speed([3, 6, 7, 11], 8) == 4
# assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
# assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23

### Bài 6 — Mini challenge: "maximize the minimum" — LeetCode 1552 (Magnetic Force Between Two Balls)

Có `n` giỏ ở các vị trí `position[i]` (đôi một khác nhau, chưa sắp xếp) trên một đường thẳng. Đặt `m` quả bóng vào `m` giỏ khác nhau sao cho **khoảng cách nhỏ nhất** giữa hai quả bóng bất kỳ là **lớn nhất**. Trả về khoảng cách đó. (`2 <= m <= n`.)

Ví dụ: `position = [1,2,3,4,7], m = 3` → `3` (đặt ở 1, 4, 7); `position = [5,4,3,2,1,1000000000], m = 2` → `999999999`.

Thử thách: bây giờ tính đơn điệu **ngược chiều** (`T...T F...F`) và ta cần tìm **`True` cuối cùng**. Viết lại mẫu binary search cho đúng — chú ý cách làm tròn `mid` để không lặp vô hạn.

In [32]:
def max_distance(position, m):
    # TODO 1: sắp xếp position
    # TODO 2: can(d): đặt bóng tham lam từ trái sang, đếm được >= m quả với khoảng cách >= d không?
    # TODO 3: lo = 1, hi = position[-1] - position[0]; tìm d LỚN NHẤT mà can(d) True
    #         gợi ý: mid = (lo + hi + 1) // 2
    pass


# assert max_distance([1, 2, 3, 4, 7], 3) == 3
# assert max_distance([5, 4, 3, 2, 1, 1000000000], 2) == 999999999

## 22. Gợi ý theo mức

**Bài 1**
- Mức 1: chạy greedy bằng tay, ghi `cur` sau mỗi phần tử.
- Mức 2: với `X = 3`, phần tử `4` đứng một mình đã vượt `3`.
- Mức 3: bản có kiểm tra trả `inf` ở `X = 3`; bản trơn trả `3` — "nói dối" vì đoạn `[4]` có tổng 4 > 3.

**Bài 2**
- Mức 1: `lo = max = 4`, `hi = sum = 15`.
- Mức 2: lần đầu `mid = 9`; greedy với 9 cho `[2,3,1,2] [4,3]`.
- Mức 3: dãy `mid` sẽ là `9 → 6 → 5`.

**Bài 3**
- Mức 1: đây chính là `count_pieces` + so sánh với `k`.
- Mức 2: kiểm tra `x > X` **trước** khi xét `cur + x > X`.
- Mức 3: khi cắt nhớ gán `cur = x` (không phải `0`).

**Bài 4**
- Mức 1: `lo = max(nums)`, `hi = sum(nums)`.
- Mức 2: `while lo < hi`, `mid = (lo + hi) // 2`.
- Mức 3: chia được → `hi = mid`; không → `lo = mid + 1`; `return lo`.

**Bài 5**
- Mức 1: `lo = 1` (ít nhất 1 quả/giờ), `hi = max(piles)` (với tốc độ đó mỗi đống mất đúng 1 giờ, tổng `len(piles) <= h` giờ).
- Mức 2: số giờ cho một đống `p` với tốc độ `v` là `ceil(p / v)`; trong số nguyên viết `(p + v - 1) // v`.
- Mức 3: `hours(v)` giảm khi `v` tăng → tìm `v` nhỏ nhất có `hours(v) <= h` bằng đúng mẫu "True đầu tiên".

**Bài 6**
- Mức 1: `can(d)` đúng thì mọi `d' < d` cũng đúng → dạng `T...T F...F`.
- Mức 2: greedy: đặt bóng đầu ở `position[0]`; mỗi giỏ tiếp theo cách quả gần nhất `>= d` thì đặt thêm.
- Mức 3: `can(mid)` → `lo = mid` (giữ mid), ngược lại `hi = mid - 1`; vì `lo = mid` nên phải làm tròn **lên** `mid = (lo + hi + 1) // 2`, nếu không khi `hi = lo + 1` sẽ kẹt mãi.

## 23. Đáp án tham khảo

### Đáp án Bài 1 và Bài 2

**Bài 1:** `count_pieces([1,4,4], X)` với `X = 3, 4, 5, 8, 9` lần lượt là `inf, 3, 2, 2, 1`. `count_pieces_naive([1,4,4], 3) = 3` là "nói dối": nó đếm được 3 đoạn `[1] [4] [4]` nhưng hai đoạn `[4]` có tổng 4 > 3 — giới hạn `X = 3` không thể đạt.

**Bài 2:** `lo = 4`, `hi = 15`.

| bước | lo | hi | mid | chia tham lam | số đoạn | cập nhật |
|---:|---:|---:|---:|---|---:|---|
| 1 | 4 | 15 | 9 | `[2,3,1,2] [4,3]` | 2 ≤ 3 | `hi = 9` |
| 2 | 4 | 9 | 6 | `[2,3,1] [2,4] [3]` | 3 ≤ 3 | `hi = 6` |
| 3 | 4 | 6 | 5 | `[2,3] [1,2] [4] [3]` | 4 > 3 | `lo = 6` |
| — | 6 | 6 | | dừng | | đáp án **6** |

3 lần gọi `can`. Cell dưới kiểm chứng cả hai bài.

In [33]:
print([count_pieces([1, 4, 4], X) for X in [3, 4, 5, 8, 9]], count_pieces_naive([1, 4, 4], 3))
assert [count_pieces([1, 4, 4], X) for X in [3, 4, 5, 8, 9]] == [float("inf"), 3, 2, 2, 1]
assert count_pieces_naive([1, 4, 4], 3) == 3

assert trace_split_array_bs([2, 3, 1, 2, 4, 3], 3) == 6

[inf, 3, 2, 2, 1] 3
nums=[2, 3, 1, 2, 4, 3], k=3: lo=max=4, hi=sum=15
bước 1: lo= 4 hi=15 mid= 9 | [[2, 3, 1, 2], [4, 3]] -> 2 đoạn <= k -> hi = 9
bước 2: lo= 4 hi= 9 mid= 6 | [[2, 3, 1], [2, 4], [3]] -> 3 đoạn <= k -> hi = 6
bước 3: lo= 4 hi= 6 mid= 5 | [[2, 3], [1, 2], [4], [3]] -> 4 đoạn >  k -> lo = 6
lo == hi == 6 -> đáp án 6



In [34]:
# Đáp án Bài 3 và Bài 4
def can_split_answer(nums, k, X):
    pieces, cur = 1, 0
    for x in nums:
        if x > X:
            return False
        if cur + x > X:
            pieces += 1
            cur = x
            if pieces > k:
                return False
        else:
            cur += x
    return True


def my_split_array_answer(nums, k):
    lo, hi = max(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if can_split_answer(nums, k, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


assert can_split_answer([7, 2, 5, 10, 8], 2, 18) is True
assert can_split_answer([7, 2, 5, 10, 8], 2, 17) is False
assert can_split_answer([1, 4, 4], 3, 3) is False
assert can_split_answer([0, 0, 0], 1, 0) is True
assert my_split_array_answer([7, 2, 5, 10, 8], 2) == 18
assert my_split_array_answer([1, 2, 3, 4, 5], 2) == 9
assert my_split_array_answer([1, 4, 4], 3) == 4

random.seed(4)
for _ in range(500):
    n = random.randint(1, 8)
    nums = [random.randint(0, 15) for _ in range(n)]
    k = random.randint(1, n)
    assert my_split_array_answer(nums, k) == split_array_bruteforce(nums, k)
print("Bài 3 + Bài 4 PASS (kèm 500 test ngẫu nhiên so với brute force)")

Bài 3 + Bài 4 PASS (kèm 500 test ngẫu nhiên so với brute force)


In [35]:
# Đáp án Bài 5 — LeetCode 875
def min_eating_speed_answer(piles, h):
    def hours(v):
        return sum((p + v - 1) // v for p in piles)      # ceil(p / v) bằng số nguyên

    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(mid) <= h:      # ăn kịp với tốc độ mid -> thử chậm hơn
            hi = mid
        else:
            lo = mid + 1
    return lo


assert min_eating_speed_answer([3, 6, 7, 11], 8) == 4
assert min_eating_speed_answer([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed_answer([30, 11, 23, 4, 20], 6) == 23

random.seed(875)
for _ in range(300):                                      # so với cách thử từng v
    piles = [random.randint(1, 30) for _ in range(random.randint(1, 6))]
    h = random.randint(len(piles), 40)
    slow = next(v for v in range(1, max(piles) + 1) if sum((p + v - 1) // v for p in piles) <= h)
    assert min_eating_speed_answer(piles, h) == slow
print("Bài 5 PASS")

Bài 5 PASS


In [36]:
# Đáp án Bài 6 — LeetCode 1552 (maximize the minimum -> tìm True CUỐI CÙNG)
def max_distance_answer(position, m):
    pos = sorted(position)

    def can(d):                         # đặt được >= m quả với khoảng cách tối thiểu d?
        count, last = 1, pos[0]
        for p in pos[1:]:
            if p - last >= d:
                count += 1
                last = p
                if count >= m:
                    return True
        return count >= m

    lo, hi = 1, pos[-1] - pos[0]
    while lo < hi:
        mid = (lo + hi + 1) // 2        # làm tròn LÊN vì nhánh đúng gán lo = mid
        if can(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


assert max_distance_answer([1, 2, 3, 4, 7], 3) == 3
assert max_distance_answer([5, 4, 3, 2, 1, 1000000000], 2) == 999999999

random.seed(1552)
for _ in range(300):                                      # so với brute force thử mọi tổ hợp giỏ
    n = random.randint(2, 7)
    position = random.sample(range(1, 40), n)
    m = random.randint(2, n)
    brute = max(min(b - a for a, b in zip(c, c[1:])) for c in combinations(sorted(position), m))
    assert max_distance_answer(position, m) == brute
print("Bài 6 PASS")

Bài 6 PASS


## 24. Template ngắn nên nhớ

```python
class Solution:
    def splitArray(self, nums: List[int], k: int) -> int:
        def can(X):                          # chia được thành <= k đoạn, mỗi đoạn <= X ?
            pieces, cur = 1, 0
            for x in nums:
                if cur + x > X:
                    pieces, cur = pieces + 1, x
                else:
                    cur += x
            return pieces <= k

        lo, hi = max(nums), sum(nums)        # cận dưới: phần tử lớn nhất; cận trên: cả mảng
        while lo < hi:                       # tìm X nhỏ nhất mà can(X) đúng
            mid = (lo + hi) // 2
            if can(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo
```

Câu thần chú: **"Đừng hỏi *bao nhiêu*, hãy hỏi *được không*. Được thì `hi = mid`, không được thì `lo = mid + 1`. Khoảng từ `max` tới `sum`. Đếm tham lam, so sánh `<= k`."**

Khuôn tổng quát cho mọi bài "minimize the maximum": **khoảng đáp án → `can(X)` đơn điệu (thường là greedy) → tìm True đầu tiên**. Với "maximize the minimum": tìm True **cuối cùng**, `mid = (lo + hi + 1) // 2`, `lo = mid` / `hi = mid - 1`.

## 25. Checklist trước khi submit

- `lo = max(nums)` (không phải `0` hay `min(nums)`) và `hi = sum(nums)`?
- Hàm kiểm tra bắt đầu `pieces = 1` (tính cả đoạn đang mở), khi cắt gán `cur = x` (không phải `0`)?
- So sánh `pieces <= k` (không phải `== k`)?
- Vòng lặp `while lo < hi` đi với `hi = mid` và `lo = mid + 1`, `mid = (lo + hi) // 2`?
- **Không** sắp xếp `nums` — đoạn phải liên tiếp theo thứ tự gốc?
- Đã thử: hai example của đề (`18`, `9`), `n = 1`, `k = 1` (ra `sum`), `k = n` (ra `max`), mảng toàn `0`, ca bẫy `[1,4,4], k = 3` (ra `4`)?
- Nói được độ phức tạp: `O(n · log(sum))` time, `O(1)` space; và vì sao DP `O(k · n²)` chậm hơn?